In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("ECommerce-Fraud-Detection")
    .master("local[*]")
    .getOrCreate()
)
# Ẩn các thông báo log rác, chỉ hiển thị thông báo lỗi hệ thống
spark.sparkContext.setLogLevel("ERROR")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/31 03:33:34 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/05/31 03:33:37 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [2]:
# Đường dẫn lưu trữ tập dữ liệu trên hệ thống tệp tin phân tán HDFS
hdfs_path = "hdfs://localhost:9090/ecommerce-fraud-detection/transactions.csv"
# Đọc dữ liệu từ HDFS vào Spark DataFrame
df = spark.read.csv(
    hdfs_path,
    header=True,
    inferSchema=True
)
dataframe = df.limit(5).toPandas()

In [3]:
df.createOrReplaceTempView("ecommerce_transactions")
spark.catalog.listTables()

[Table(name='ecommerce_transactions', catalog=None, namespace=[], description=None, tableType='TEMPORARY', isTemporary=True)]

In [4]:
# Tỷ lệ và tổng giá trị của giao dịch gian lận so với giao dịch hợp lệ là bao nhiêu?
query_1 = spark.sql("""
    SELECT
        CASE
            WHEN is_fraud = 1 THEN 'Gian lận (Fraudulent)'
            ELSE 'Hợp lệ (Legitimate)'
        END AS loai_giao_dich,
        COUNT(*) AS so_luong_giao_dich,
        CAST(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER() AS DECIMAL(5, 2)) AS ty_le_phan_tram,
        SUM(amount) AS tong_gia_tri,
        AVG(amount) AS gia_tri_trung_binh
    FROM ecommerce_transactions
    GROUP BY is_fraud;
""")
query_1.toPandas().style.hide(axis="index")

loai_giao_dich,so_luong_giao_dich,ty_le_phan_tram,tong_gia_tri,gia_tri_trung_binh
Gian lận (Fraudulent),6612,2.21,3907435.450000,590.961199
Hợp lệ (Legitimate),293083,97.79,49188112.710000,167.829976


In [5]:
query_2 = spark.sql("""
    SELECT
        merchant_category,
        COUNT(*) AS so_luong_gian_lan,
        SUM(amount) AS tong_thiet_hai
    FROM ecommerce_transactions
    WHERE is_fraud = 1
    GROUP BY merchant_category
    ORDER BY tong_thiet_hai DESC
""")
query_2.toPandas().style.hide(axis="index")

merchant_category,so_luong_gian_lan,tong_thiet_hai
electronics,1357,840480.360000
travel,1388,786641.000000
fashion,1340,775658.030000
gaming,1295,761831.010000
grocery,1232,742825.050000


In [6]:
query_3 = spark.sql("""
    WITH FraudStats AS (
        SELECT
            CASE
                WHEN cvv_result = 1 THEN 'CVV Đúng (Correct)'
                WHEN cvv_result = 0 THEN 'CVV Sai (Incorrect)'
                ELSE 'Không rõ'
            END AS ket_qua_cvv,
            CASE
                WHEN three_ds_flag = 1 THEN 'Có 3DS'
                ELSE 'Không có 3DS'
            END AS xac_thuc_3ds,
            is_fraud
        FROM ecommerce_transactions
    )
    SELECT
        ket_qua_cvv,
        xac_thuc_3ds,
        COUNT(*) AS tong_giao_dich,
        SUM(is_fraud) AS so_giao_dich_gian_lan,
        CAST(SUM(is_fraud) * 100.0 / COUNT(*) AS DECIMAL(5, 2)) AS ty_le_gian_lan
    FROM FraudStats
    GROUP BY ket_qua_cvv, xac_thuc_3ds
    ORDER BY ket_qua_cvv, xac_thuc_3ds;
""")
query_3.toPandas().style.hide(axis="index")

ket_qua_cvv,xac_thuc_3ds,tong_giao_dich,so_giao_dich_gian_lan,ty_le_gian_lan
CVV Sai (Incorrect),Có 3DS,10758,404,3.76
CVV Sai (Incorrect),Không có 3DS,27570,3661,13.28
CVV Đúng (Correct),Có 3DS,224379,1849,0.82
CVV Đúng (Correct),Không có 3DS,36988,698,1.89


In [7]:
query_4 = spark.sql("""
    SELECT
        HOUR(transaction_time) AS khung_gio_trong_ngay,
        country,
        SUM(CASE WHEN is_fraud = 1 THEN 1 ELSE 0 END) AS so_luong_gian_lan,
        COUNT(*) AS tong_giao_dich,
        CAST(SUM(CASE WHEN is_fraud = 1 THEN 1 ELSE 0 END) * 100.0 / COUNT(*) AS DECIMAL(5, 2)) AS ty_le_gian_lan
    FROM ecommerce_transactions
    WHERE country IS NOT NULL AND country != 'Unknown'
    GROUP BY khung_gio_trong_ngay, country
    ORDER BY so_luong_gian_lan DESC
    LIMIT 10;
""")
query_4.toPandas().style.hide(axis="index")

khung_gio_trong_ngay,country,so_luong_gian_lan,tong_giao_dich,ty_le_gian_lan
23,TR,47,1213,3.87
10,TR,43,1264,3.40
8,US,42,1349,3.11
11,TR,42,1231,3.41
19,GB,41,1320,3.11
7,TR,41,1258,3.26
17,PL,40,1208,3.31
16,TR,40,1197,3.34
9,TR,40,1155,3.46
11,PL,40,1291,3.10


In [8]:
query_5 = spark.sql("""
    SELECT
        CASE
            WHEN account_age_days <= 30 THEN '0-30 ngày (Tài khoản mới)'
            WHEN account_age_days > 30 AND account_age_days <= 180 THEN '31-180 ngày (Tương đối mới)'
            WHEN account_age_days > 180 AND account_age_days <= 365 THEN '181-365 ngày (Tương đối cũ)'
            ELSE '> 365 ngày (Tài khoản cũ)'
        END AS nhom_tuoi_tai_khoan,
        COUNT(*) AS tong_giao_dich,
        SUM(is_fraud) AS so_giao_dich_gian_lan,
        CAST(SUM(is_fraud) * 100.0 / COUNT(*) AS DECIMAL(5, 2)) AS ty_le_gian_lan
    FROM ecommerce_transactions
    GROUP BY nhom_tuoi_tai_khoan
    ORDER BY ty_le_gian_lan DESC;
""")
query_5.toPandas().style.hide(axis="index")


nhom_tuoi_tai_khoan,tong_giao_dich,so_giao_dich_gian_lan,ty_le_gian_lan
0-30 ngày (Tài khoản mới),2081,1051,50.50
31-180 ngày (Tương đối mới),17512,2205,12.59
> 365 ngày (Tài khoản cũ),250488,3008,1.20
181-365 ngày (Tương đối cũ),29614,348,1.18


In [9]:
query_6 = spark.sql("""
    SELECT
        CASE
            WHEN amount > (avg_amount_user * 5) THEN 'Bất thường (Amount > 5 * Avg)'
            ELSE 'Bình thường'
        END AS loai_chi_tieu,
        COUNT(*) AS tong_giao_dich,
        SUM(is_fraud) AS so_giao_dich_gian_lan,
        CAST(SUM(is_fraud) * 100.0 / COUNT(*) AS DECIMAL(5, 2)) AS ty_le_gian_lan
    FROM ecommerce_transactions
    WHERE  avg_amount_user > 0
    GROUP BY loai_chi_tieu;
""")
query_6.toPandas().style.hide(axis="index")


loai_chi_tieu,tong_giao_dich,so_giao_dich_gian_lan,ty_le_gian_lan
Bất thường (Amount > 5 * Avg),1673,1503,89.84
Bình thường,298022,5109,1.71


In [10]:
query_7 = spark.sql("""
    SELECT
        CASE
            WHEN country = bin_country THEN 'Trùng khớp (Matched)'
            ELSE 'Không trùng khớp (Mismatched)'
        END AS trang_thai_dia_ly,
        COUNT(*) AS tong_giao_dich,
        SUM(is_fraud) AS so_giao_dich_gian_lan,
        CAST(SUM(is_fraud) * 100.0 / COUNT(*) AS DECIMAL(5, 2)) AS ty_le_gian_lan
    FROM ecommerce_transactions
    WHERE
        country IS NOT NULL AND bin_country IS NOT NULL
        AND country != 'Unknown' AND bin_country != 'Unknown'
    GROUP BY trang_thai_dia_ly;
""")
query_7.toPandas().style.hide(axis="index")


trang_thai_dia_ly,tong_giao_dich,so_giao_dich_gian_lan,ty_le_gian_lan
Trùng khớp (Matched),275965,3936,1.43
Không trùng khớp (Mismatched),23730,2676,11.28


In [11]:
query_8 = spark.sql("""
SELECT
    CASE
        WHEN promo_used = 1 THEN 'Có sử dụng khuyến mãi'
        ELSE 'Không sử dụng khuyến mãi'
    END AS promotion_status,
    COUNT(*) AS total_transactions,
    SUM(is_fraud) AS fraud_transactions,
    ROUND(
        SUM(is_fraud) * 100.0 / COUNT(*), 2
    ) AS fraud_rate,
    ROUND(AVG(amount), 2) AS average_amount
FROM ecommerce_transactions
GROUP BY promo_used
ORDER BY fraud_rate DESC
""")
query_8.toPandas().style.hide(axis="index")

promotion_status,total_transactions,fraud_transactions,fraud_rate,average_amount
Có sử dụng khuyến mãi,46045,2085,4.53,187.100000
Không sử dụng khuyến mãi,253650,4527,1.78,175.360000


In [12]:
query_9 = spark.sql("""
SELECT
    CASE
        WHEN shipping_distance_km <= 50 THEN '0-50 km'
        WHEN shipping_distance_km <= 500 THEN '51-500 km'
        WHEN shipping_distance_km <= 2000 THEN '501-2000 km'
        ELSE '>2000 km'
    END AS shipping_group,
    COUNT(*) AS total_transactions,
    SUM(is_fraud) AS fraud_transactions,
    ROUND(SUM(is_fraud) * 100.0 / COUNT(*), 2) AS fraud_rate,
    ROUND(AVG(amount), 2) AS average_amount
FROM ecommerce_transactions

GROUP BY
    CASE
        WHEN shipping_distance_km <= 50 THEN '0-50 km'
        WHEN shipping_distance_km <= 500 THEN '51-500 km'
        WHEN shipping_distance_km <= 2000 THEN '501-2000 km'
        ELSE '>2000 km'
    END
ORDER BY fraud_rate DESC
""")

query_9.toPandas().style.hide(axis="index")

shipping_group,total_transactions,fraud_transactions,fraud_rate,average_amount
>2000 km,6864,1535,22.36,263.120000
501-2000 km,18167,2443,13.45,222.260000
51-500 km,247239,2384,0.96,171.890000
0-50 km,27425,250,0.91,173.330000


In [13]:
query_10 = spark.sql("""
WITH MerchantStats AS (
    SELECT
        merchant_category,
        COUNT(*) AS total_transactions,
        SUM(is_fraud) AS fraud_transactions,
        ROUND( SUM(is_fraud) * 100.0 / COUNT(*), 2) AS fraud_rate
    FROM ecommerce_transactions
    GROUP BY merchant_category
    HAVING COUNT(*) >= 100
)

SELECT
    RANK() OVER (ORDER BY fraud_rate DESC) AS ranking,
    merchant_category, total_transactions, fraud_transactions, fraud_rate
FROM MerchantStats
ORDER BY ranking
""")

query_10.toPandas().style.hide(axis="index")

ranking,merchant_category,total_transactions,fraud_transactions,fraud_rate
1,travel,59922,1388,2.32
2,electronics,60220,1357,2.25
3,fashion,59801,1340,2.24
4,gaming,59839,1295,2.16
5,grocery,59913,1232,2.06


In [14]:
query_11 = spark.sql("""
WITH ChannelStats AS (
    SELECT channel,
        COUNT(*) AS total_transactions,
        SUM(is_fraud) AS fraud_transactions,
        ROUND(SUM(is_fraud) * 100.0 / COUNT(*), 2) AS fraud_rate,
        ROUND(AVG(amount), 2) AS average_amount
    FROM ecommerce_transactions
    GROUP BY channel
)

SELECT
    RANK() OVER (ORDER BY fraud_rate DESC) AS ranking,
    channel, total_transactions, fraud_transactions, fraud_rate, average_amount
FROM ChannelStats
ORDER BY ranking
""")

query_11.toPandas().style.hide(axis="index")

ranking,channel,total_transactions,fraud_transactions,fraud_rate,average_amount
1,web,152226,5426,3.56,182.670000
2,app,147469,1186,0.80,171.480000


In [15]:
query_12 = spark.sql("""
SELECT
    CASE
        WHEN DAYOFWEEK(transaction_time) = 2 THEN 'Monday'
        WHEN DAYOFWEEK(transaction_time) = 3 THEN 'Tuesday'
        WHEN DAYOFWEEK(transaction_time) = 4 THEN 'Wednesday'
        WHEN DAYOFWEEK(transaction_time) = 5 THEN 'Thursday'
        WHEN DAYOFWEEK(transaction_time) = 6 THEN 'Friday'
        WHEN DAYOFWEEK(transaction_time) = 7 THEN 'Saturday'
        ELSE 'Sunday'
    END AS day_of_week,

    CASE
        WHEN amount < 50 THEN 'Under 50 USD'
        WHEN amount < 200 THEN '50 - 200 USD'
        WHEN amount < 500 THEN '200 - 500 USD'
        WHEN amount < 1000 THEN '500 - 1000 USD'
    ELSE 'Over 1000 USD'
    END AS amount_group,
    
    COUNT(*) AS total_transactions,
    SUM(is_fraud) AS fraud_transactions,
    ROUND(SUM(is_fraud) * 100.0 / COUNT(*), 2) AS fraud_rate,
    ROUND(AVG(amount), 2) AS average_amount

FROM ecommerce_transactions

GROUP BY DAYOFWEEK(transaction_time),
    CASE
        WHEN DAYOFWEEK(transaction_time) = 2 THEN 'Monday'
        WHEN DAYOFWEEK(transaction_time) = 3 THEN 'Tuesday'
        WHEN DAYOFWEEK(transaction_time) = 4 THEN 'Wednesday'
        WHEN DAYOFWEEK(transaction_time) = 5 THEN 'Thursday'
        WHEN DAYOFWEEK(transaction_time) = 6 THEN 'Friday'
        WHEN DAYOFWEEK(transaction_time) = 7 THEN 'Saturday'
        ELSE 'Sunday'
    END,

    CASE
        WHEN amount < 50 THEN 'Under 50 USD'
        WHEN amount < 200 THEN '50 - 200 USD'
        WHEN amount < 500 THEN '200 - 500 USD'
        WHEN amount < 1000 THEN '500 - 1000 USD'
        ELSE 'Over 1000 USD'
    END

HAVING COUNT(*) >= 30
ORDER BY DAYOFWEEK(transaction_time) ASC,
    fraud_rate DESC, fraud_transactions DESC
""")
query_12.toPandas().style.hide(axis="index")


day_of_week,amount_group,total_transactions,fraud_transactions,fraud_rate,average_amount
Sunday,Over 1000 USD,895,254,28.38,1765.450000
Sunday,Under 50 USD,12662,328,2.59,27.670000
Sunday,500 - 1000 USD,1922,32,1.66,679.090000
Sunday,50 - 200 USD,19546,245,1.25,106.370000
Sunday,200 - 500 USD,7085,76,1.07,303.370000
Monday,Over 1000 USD,939,244,25.99,1714.780000
Monday,Under 50 USD,12973,329,2.54,27.720000
Monday,50 - 200 USD,19691,266,1.35,106.490000
Monday,500 - 1000 USD,1963,26,1.32,676.720000
Monday,200 - 500 USD,7369,95,1.29,305.690000


In [16]:
query_13 = spark.sql("""
WITH HourlyTransactions AS (
SELECT user_id,
    DATE_TRUNC('hour', transaction_time) AS transaction_hour,
    COUNT(*) AS transaction_count,
    SUM(is_fraud) AS fraud_transactions,
    ROUND(AVG(amount), 2) AS average_amount
FROM ecommerce_transactions
GROUP BY user_id,
    DATE_TRUNC('hour', transaction_time)
)

SELECT
    ROW_NUMBER() OVER (ORDER BY transaction_count DESC, fraud_transactions DESC) AS ranking,
    user_id, transaction_hour, transaction_count, fraud_transactions, average_amount
FROM HourlyTransactions

ORDER BY transaction_count DESC, fraud_transactions DESC
LIMIT 20
""")
query_13.toPandas().style.hide(axis="index")

ranking,user_id,transaction_hour,transaction_count,fraud_transactions,average_amount
1,3980,2024-09-09 01:00:00,3,0,28.370000
2,1925,2024-06-19 18:00:00,3,0,1279.080000
3,1990,2024-02-03 12:00:00,2,2,1212.440000
4,4444,2024-10-08 16:00:00,2,2,2.270000
5,95,2024-08-24 13:00:00,2,2,80.980000
6,3370,2024-10-02 04:00:00,2,2,1910.950000
7,448,2024-01-08 04:00:00,2,1,742.260000
8,1200,2024-04-24 16:00:00,2,1,236.670000
9,1277,2024-01-08 16:00:00,2,1,824.040000
10,3236,2024-05-09 23:00:00,2,1,69.940000


In [ ]:
query_14 = spark.sql("""
WITH UserBehavior AS (
    SELECT
        user_id,
        COUNT(*) AS total_transactions,
        SUM(is_fraud) AS fraud_transactions,
        AVG(shipping_distance_km) AS avg_distance,
        SUM(
            CASE
                WHEN cvv_result = 0 THEN 1
                ELSE 0
            END
        ) AS failed_cvv_attempts,
        SUM(
            CASE
                WHEN country != bin_country THEN 1
                ELSE 0
            END
        ) AS geographic_mismatch,
       MAX(amount) / MAX(avg_amount_user) AS spending_spike_ratio
    FROM ecommerce_transactions
    WHERE avg_amount_user > 0
    GROUP BY user_id
),

RiskScored AS (
    SELECT *,
    (
       (CAST(fraud_transactions AS DOUBLE) / total_transactions * 40) + (failed_cvv_attempts * 1.5) + (geographic_mismatch * 2.0) + (LEAST(spending_spike_ratio * 3, 30))
    ) AS risk_score
    FROM UserBehavior
)

SELECT
    DENSE_RANK() OVER (
        ORDER BY risk_score DESC
    ) AS ranking,
    user_id, total_transactions, fraud_transactions,
    ROUND(avg_distance, 1) AS avg_distance_km,
    failed_cvv_attempts,
    geographic_mismatch,
    ROUND(spending_spike_ratio, 2) AS spending_spike_ratio,
    ROUND(risk_score, 2) AS risk_score
FROM RiskScored

ORDER BY risk_score DESC
LIMIT 10
""")
query_14.toPandas().style.hide(axis="index")

ranking,user_id,total_transactions,fraud_transactions,avg_distance_km,failed_cvv_attempts,geographic_mismatch,spending_spike_ratio,risk_score
1,1093,56,33,936.500000,25,20,15.310000,131.070000
2,5918,60,33,818.100000,25,19,205.750000,127.500000
3,5754,58,32,860.800000,24,18,122.550000,124.070000
4,5058,59,34,796.000000,27,14,31.950000,121.550000
5,3140,56,31,811.800000,24,16,43.220000,120.140000
6,4449,52,30,974.500000,23,16,20.190000,119.580000
7,381,58,31,906.600000,21,18,76.180000,118.880000
8,4873,54,30,857.800000,23,16,112.180000,118.720000
9,291,59,30,759.500000,25,15,37.730000,117.840000
10,2572,58,33,702.000000,26,13,130.120000,117.760000


26/05/31 10:25:39 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:53)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:359)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:36)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:132)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$$driverEndpoint(BlockManagerMasterEndpoint.scala:131)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.isExecutorAlive$lzycompute$1(BlockManagerMasterEndpoint.scala:707)
	at org.apache.spark.storage.BlockManagerMasterE